In [1]:
import json
import os
import re

from num2words import num2words

In [2]:
_PUNCT_PATTERN = re.compile(r"[^\w\s-]")
_WHITESPACE_PATTERN = re.compile(r"\s+")
_NUMBER_PATTERN = re.compile(r"\d+")


def remove_punctuation(text: str) -> str:
    return _PUNCT_PATTERN.sub(" ", text)


def normalize_numbers(text: str, lang: str = "id") -> str:
    return _NUMBER_PATTERN.sub(lambda m: num2words(int(m.group()), lang=lang), text)


def normalize(text: str) -> str:
    text = remove_punctuation(text)
    text = normalize_numbers(text)
    text = _WHITESPACE_PATTERN.sub(" ", text).strip().lower()
    return text

In [3]:
DATA_PATH = os.path.join("..", "data", "raw", "augmented_ordering_dataset.jsonl")

records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"loaded {len(records)} records from {DATA_PATH}")

loaded 1943 records from ../data/raw/augmented_ordering_dataset.jsonl


In [4]:
for r in records:
    r["text_normalized"] = normalize(r["text"])

changed = [r for r in records if r["text"].strip().lower() != r["text_normalized"]]
print(f"{len(changed)} of {len(records)} rows changed by normalization\n")

for r in changed[:10]:
    print("RAW: ", r["text"])
    print("NORM:", r["text_normalized"])
    print()

1943 of 1943 rows changed by normalization

RAW:  eh saya mau, eh yaudah tambah Nasi Campur satu lagi, eh saya mau juga Ayam Bakar satu, eh yaudah deh
NORM: eh saya mau eh yaudah tambah nasi campur satu lagi eh saya mau juga ayam bakar satu eh yaudah deh

RAW:  eh saya mau, eh yaudah tambah Nasi Rames satu lagi, eh saya mau juga Ayam Geprek satu, eh yaudah deh
NORM: eh saya mau eh yaudah tambah nasi rames satu lagi eh saya mau juga ayam geprek satu eh yaudah deh

RAW:  eh saya mau, eh yaudah tambah Sup Ayam satu lagi, eh saya mau juga Ayam Bakar satu, eh yaudah deh
NORM: eh saya mau eh yaudah tambah sup ayam satu lagi eh saya mau juga ayam bakar satu eh yaudah deh

RAW:  eh, Kalau... eh, mau tahu harga Nasi Goreng Kambing ya, maksudnya, oh iya juga bisa ngasih rekomendasi makanan lainnya, mungkin Sup Buntut?
NORM: eh kalau eh mau tahu harga nasi goreng kambing ya maksudnya oh iya juga bisa ngasih rekomendasi makanan lainnya mungkin sup buntut

RAW:  eh, Kalau... eh, mau tahu harga Bakm

In [5]:
OUTPUT_PATH = os.path.join(
    "..", "data", "normalized", "augmented_ordering_dataset_normalized.jsonl"
)

os.makedirs("../data/normalized", exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"wrote {len(records)} normalized records to {OUTPUT_PATH}")

wrote 1943 normalized records to ../data/normalized/augmented_ordering_dataset_normalized.jsonl
